# FactorJoin Reproduction

This notebook reproduces FactorJoin on `STATS-CEB`, `JOBLight`, `JOBLightRanges`, `JOBM`, and `JobJoin`.

It records raw timing components separately:
- `train_time_sec`
- `prepare_sample_time_sec`
- `materialize_sample_time_sec`
- `estimate_time_sec`
- `total_build_like_time_sec`
- `total_eval_like_time_sec`

Important notes:
- `STATS` and `JOBLight` use BN mode.
- `JOBM` uses sampling mode and must be evaluated on grouped subqueries before remapping back to `subquery.sql` order.
- `JobJoin` is the full 21-table, predicate-free join workload. It is evaluated at the **main-query level** (31 queries) via the fixed `get_cardinality_bound_one` path (not the LpBound `get_cardinality_bound_all`). Being predicate-free, it needs no PostgreSQL: training reads CSV only, materialization yields all-`None` samples, and estimation falls back to `ground_truth_factors_no_filter`. The cyclic self-join query (Q31) is architecturally unsupported by FactorJoin and is written as `MISSING`. Output goes to `Benchmark/workloads/JobJoin/result/factorjoin.txt` (the file `EvaluateAccuracy` reads).
- In the current FactorJoin implementation, `sampling_percentage` is interpreted in percent units, so `1.0` means **1 percent**.
- This notebook raises on missing `JOBM` artifacts instead of silently filling fallback values.


In [1]:
from __future__ import annotations

import os
import pickle
import shutil
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2 as pg

PROJECT_ROOT = (Path.cwd().parent if (Path.cwd().parent / "Benchmark").exists() else Path.cwd()).resolve()
EXPERIMENT_DIR = PROJECT_ROOT / "experiment"
WORKLOAD_DIR = PROJECT_ROOT / "Benchmark" / "workloads"
FACTORJOIN_DIR = PROJECT_ROOT / "methods" / "FactorJoin"
FACTORJOIN_CHECKPOINT_DIR = FACTORJOIN_DIR / "checkpoints"
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoint" / "FactorJoin"
SUMMARY_CSV_PATH = CHECKPOINT_DIR / "benchmark_times.csv"

FACTORJOIN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(FACTORJOIN_DIR))

from Evaluation.training import train_one_stats
from Join_scheme.binning import identify_key_values
from Join_scheme.bound import Bound_ensemble
from Join_scheme.data_prepare import preprocess_imdb_light_data, process_imdb_data
from Sampling.create_binned_cols import create_binned_cols
from Sampling.get_query_binned_cards import get_query_binned_cards
from Schemas.imdb.schema import gen_jobm_schema

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"CHECKPOINT_DIR={CHECKPOINT_DIR}")

/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.4) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


PROJECT_ROOT=/home/liwei/starce-final/StarCE
CHECKPOINT_DIR=/home/liwei/starce-final/StarCE/experiment/checkpoint/FactorJoin


/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/liwei/miniconda3/envs/TestEnv/lib/python3.10/site-packages/pgmpy/utils/utils.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
BENCHMARKS_TO_RUN = ["STATS", "JOBLight", "JOBLightRanges", "JOBM"]
if "StatsJoin" not in BENCHMARKS_TO_RUN:
    BENCHMARKS_TO_RUN = BENCHMARKS_TO_RUN + ["StatsJoin"]
FORCE_REPARSE_JOBM_GROUPED = True
JOBM_SAMPLING_PERCENTAGE = 1.0  # FactorJoin interprets this in percent units, so this means 1%.
JOBM_SAMPLING_TYPE = "ss"
JOBM_SAMPLE_SIZE = 1_000_000
JOBM_DB_CONN_KWARGS = "dbname=imdbm host=localhost port=5432"
SEED = 0

STATS_CFG = {
    "dataset": "stats",
    "data_path": str(PROJECT_ROOT / "methods" / "SafeBound" / "Data" / "Stats" / "{}.csv"),
    "query_file": WORKLOAD_DIR / "STATS-CEB" / "subquery" / "subquery.sql",
    "model_path": FACTORJOIN_CHECKPOINT_DIR / "model_stats_greedy_200.pkl",
    "raw_output": CHECKPOINT_DIR / "raw_stats_CEB_sub_queries_model_stats_greedy_200.txt",
    "checkpoint_output": CHECKPOINT_DIR / "card_stats.txt",
    "n_dim_dist": 2,
    "n_bins": 200,
    "bucket_method": "greedy",
    "get_bin_means": False,
}

JOBLIGHT_CFG = {
    "dataset": "imdb-light",
    "data_path": str(PROJECT_ROOT / "methods" / "SafeBound" / "Data" / "IMDB" / "{}.csv"),
    "query_file": WORKLOAD_DIR / "JOBLight" / "subquery" / "subquery.sql",
    "true_card_file": WORKLOAD_DIR / "JOBLight" / "subquery" / "result" / "real.txt",
    "model_path": FACTORJOIN_CHECKPOINT_DIR / "model_imdb-light_fixed_start_key_200.pkl",
    "raw_output": CHECKPOINT_DIR / "raw_factorjoin_joblight.txt",
    "checkpoint_output": CHECKPOINT_DIR / "card_joblight.txt",
    "n_dim_dist": 2,
    "n_bins": 200,
    "bucket_method": "fixed_start_key",
    "get_bin_means": True,
}

JOBLIGHTRANGES_CFG = {
    "dataset": "imdb-light",
    "raw_data_path": str(PROJECT_ROOT / "methods" / "SafeBound" / "Data" / "IMDB" / "{}.csv"),
    "raw_query_file": WORKLOAD_DIR / "JOBLightRanges" / "subquery" / "subquery.sql",
    "true_card_file": WORKLOAD_DIR / "JOBLightRanges" / "subquery" / "result" / "real.txt",
    "preprocessed_dir": FACTORJOIN_CHECKPOINT_DIR / "joblightranges_preprocessed",
    "preprocessed_query_file": FACTORJOIN_CHECKPOINT_DIR / "joblightranges_preprocessed" / "JOBLightRangesQueries.sql",
    "model_dir": FACTORJOIN_CHECKPOINT_DIR / "joblightranges_models",
    "model_path": FACTORJOIN_CHECKPOINT_DIR / "joblightranges_models" / "model_imdb-light_fixed_start_key_200.pkl",
    "raw_output": CHECKPOINT_DIR / "raw_factorjoin_joblr.txt",
    "checkpoint_output": CHECKPOINT_DIR / "card_joblr.txt",
    "n_dim_dist": 2,
    "n_bins": 200,
    "bucket_method": "fixed_start_key",
    "get_bin_means": True,
}

JOBM_CFG = {
    "dataset": "jobm",
    "data_path": str(PROJECT_ROOT / "methods" / "SafeBound" / "Data" / "IMDB" / "{}.csv"),
    "queries_sql": WORKLOAD_DIR / "JOBM" / "queries.sql",
    "grouped_file": CHECKPOINT_DIR / "subquery_grouped.sql",
    "grouped_subquery_file": WORKLOAD_DIR / "JOBM" / "subquery" / "subquery_from_grouped.sql",
    "target_subquery_file": WORKLOAD_DIR / "JOBM" / "subquery" / "subquery.sql",
    "mapping_file": FACTORJOIN_CHECKPOINT_DIR / "jobm_sub_to_main.pkl",
    "main_queries_dir": FACTORJOIN_CHECKPOINT_DIR / "jobm_main_queries",
    "model_path": FACTORJOIN_CHECKPOINT_DIR / "model_jobm_default.pkl",
    "grouped_output": CHECKPOINT_DIR / "raw_factorjoin_jobm_grouped.txt",
    "checkpoint_output": CHECKPOINT_DIR / "card_jobm.txt",
    "sampling_percentage": JOBM_SAMPLING_PERCENTAGE,
    "sampling_type": JOBM_SAMPLING_TYPE,
    "sample_size": JOBM_SAMPLE_SIZE,
    "db_conn_kwargs": JOBM_DB_CONN_KWARGS,
    "seed": SEED,
    # Paths for generate_jobm_grouped_file (StarCE run)
    "starce_binary": PROJECT_ROOT / "build" / "starce",
    "schema_path": PROJECT_ROOT / "Benchmark" / "IMDB" / "schema_imdb.json",
    "db_path": PROJECT_ROOT / "Benchmark" / "duckdb" / "imdb.db",
    "stats_path": PROJECT_ROOT / "experiment" / "checkpoint" / "StarCE" / "statistics_imdb.json",
    "single_query_path": WORKLOAD_DIR / "JOBM" / "single_query" / "single_query.sql",
    "single_query_result_path": WORKLOAD_DIR / "JOBM" / "single_query" / "pg_est.txt",
}


# ---- StatsJoin (STATS-CEB predicate-free pure join; BN mode, subquery-level evaluation) ----
# Reuses stats model model_stats_greedy_200.pkl, directly runs BN inference on StatsJoin subqueries.
# No predicates -> BN inference only computes join upper bound, no sampling needed, no PG.
STATSJOIN_CFG = {
    "dataset": "stats",
    "data_path": str(PROJECT_ROOT / "methods" / "SafeBound" / "Data" / "Stats" / "{}.csv"),
    "query_file": WORKLOAD_DIR / "StatsJoin" / "subquery" / "subquery.sql",
    "model_path": FACTORJOIN_CHECKPOINT_DIR / "model_stats_greedy_200.pkl",
    "raw_output": CHECKPOINT_DIR / "raw_statsjoin_sub_queries_model_stats_greedy_200.txt",
    "checkpoint_output": CHECKPOINT_DIR / "card_statsjoin.txt",
    "result_output": WORKLOAD_DIR / "StatsJoin" / "subquery" / "result" / "factorjoin.txt",
    "n_dim_dist": 2,
    "n_bins": 200,
    "bucket_method": "greedy",
    "get_bin_means": False,
}


In [3]:
# ---- JobJoin (full 21 tables, predicate-free pure join; main-query-level evaluation) ----
# Use the fixed get_cardinality_bound_one path (different from LpBound's get_cardinality_bound_all).
# No predicates -> training reads CSV only, materialization doesn't connect PG, estimation falls back to ground_truth_factors_no_filter.
# Cyclic self-join query (Q31) is architecturally unsupported by FactorJoin, written as MISSING during evaluation (no fallback fake data).
if "JobJoin" not in BENCHMARKS_TO_RUN:
    BENCHMARKS_TO_RUN = BENCHMARKS_TO_RUN + ["JobJoin"]
JOBJOIN_DB_CONN_KWARGS = "dbname=imdb host=localhost port=5432"  # Not actually used without predicates, kept as fallback

JOBJOIN_CFG = {
    "dataset": "imdb",  # Full 21-table gen_imdb_schema, not jobm 17 tables
    "data_path": str(PROJECT_ROOT / "methods" / "SafeBound" / "Data" / "IMDB" / "{}.csv"),
    "queries_sql": WORKLOAD_DIR / "JobJoin" / "queries.sql",
    "main_queries_dir": FACTORJOIN_CHECKPOINT_DIR / "jobjoin_main_queries",
    "clean_query_file": FACTORJOIN_CHECKPOINT_DIR / "jobjoin_queries_clean.sql",
    "mapping_file": FACTORJOIN_CHECKPOINT_DIR / "jobjoin_sub_to_main.pkl",
    "model_path": FACTORJOIN_CHECKPOINT_DIR / "model_imdb_default.pkl",
    "gt_cache": FACTORJOIN_CHECKPOINT_DIR / "gt_no_filter.pkl",  # No dataset name, shared by jobm/imdb, must delete before training
    "result_output": WORKLOAD_DIR / "JobJoin" / "result" / "factorjoin.txt",  # EvaluateAccuracy reads this file
    "checkpoint_output": CHECKPOINT_DIR / "card_jobjoin.txt",
    "n_dim_dist": 1,
    "n_bins": None,  # train_one_imdb internally parses imdb default binning
    "bucket_method": "fixed_start_key",
    "sampling_percentage": 1.0,
    "sampling_type": "ss",
    "sample_size": JOBM_SAMPLE_SIZE,
    "db_conn_kwargs": JOBJOIN_DB_CONN_KWARGS,
    "seed": SEED,
}


In [4]:
def require_file(path: Path) -> Path:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


def require_dir(path: Path) -> Path:
    path = Path(path)
    if not path.is_dir():
        raise FileNotFoundError(f"Required directory not found: {path}")
    return path


def non_empty_lines(path: Path) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def count_non_empty_lines(path: Path) -> int:
    return len(non_empty_lines(path))


def dump_predictions(predictions: list[float], output_path: Path) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for pred in predictions:
            f.write(f"{pred}\n")
    return output_path


def dump_time_file(latencies: list[float], time_file: Path) -> Path:
    time_file.parent.mkdir(parents=True, exist_ok=True)
    with open(time_file, "w", encoding="utf-8") as f:
        for lat in latencies:
            f.write(f"{lat}\n")
    return time_file


def copy_result_file(src: Path, dst: Path) -> Path:
    src = require_file(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dst)
    return dst


def file_size_bytes(path: Path) -> int:
    path = require_file(path)
    return int(path.stat().st_size)


def dir_size_bytes(path: Path) -> int:
    path = require_dir(path)
    total = 0
    for f in path.rglob("*"):
        if f.is_file():
            total += f.stat().st_size
    return total


def measure_call(func, *args, **kwargs):
    start = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - start
    return result, elapsed


def load_bound_ensemble(model_path: Path):
    with open(model_path, "rb") as f:
        return pickle.load(f)


def estimate_bn_queries(model_path: Path, query_file: Path, output_path: Path) -> dict:
    model_path = require_file(model_path)
    query_file = require_file(query_file)

    bound_ensemble = load_bound_ensemble(model_path)
    for table in bound_ensemble.bns:
        bound_ensemble.bns[table].init_inference_method()

    queries = non_empty_lines(query_file)
    predictions = []
    latencies = []

    # Pre-parse all queries (not counted towards estimation time)
    pre_parsed = []
    for query_str in queries:
        query = query_str.split("||")[0].strip()
        pre_parsed.append(bound_ensemble.parse_query_simple(query))

    for query_str, parsed in zip(queries, pre_parsed):
        query = query_str.split("||")[0].strip()
        t0 = time.time()
        predictions.append(bound_ensemble.get_cardinality_bound_one(query, parsed=parsed))
        latencies.append(time.time() - t0)

    dump_predictions(predictions, output_path)
    time_file = output_path.parent / output_path.name.replace("card_", "estimate_time_").replace(".txt", "")
    if "raw_" in str(output_path):
        time_file = output_path.parent / ("estimate_time_" + str(output_path.stem).replace("raw_factorjoin_", "").replace("_grouped", "") + ".txt")
    else:
        bench = str(output_path.stem).replace("card_", "")
        bench_display = {"stats": "STATS", "joblight": "JOBLight", "joblr": "JOBLightRanges", "jobm": "JOBM"}.get(bench, bench.upper())
        time_file = output_path.parent / f"estimate_time_{bench_display}.txt"
    dump_time_file(latencies, time_file)

    return {
        "query_count": len(queries),
        "average_latency_sec": float(np.mean(latencies)) if latencies else 0.0,
        "prediction_path": str(output_path),
        "time_file": str(time_file),
    }


def generate_jobm_grouped_file(
    queries_sql: Path,
    output_path: Path,
    starce_binary: Path,
    schema_path: Path,
    db_path: Path,
    stats_path: Path,
    single_query_path: Path,
    single_query_result_path: Path,
) -> Path:
    """Generate subquery_grouped.sql using StarCE RecordingSubquery + SubqueryOutputGroupByMain mode.

    EXPLAINs JOBM main queries one by one, StarCE records all encountered subqueries in the optimizer,
    grouped output by === N ===. Generated file placed in checkpoint directory, not gitignored.
    """
    import json
    import subprocess
    import tempfile

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    require_file(starce_binary)
    require_file(queries_sql)
    require_file(schema_path)
    require_file(db_path)
    require_file(stats_path)

    work_dir = Path(tempfile.mkdtemp(prefix="starce_jobm_grouped_"))
    try:
        explain_path = work_dir / "explain.sql"
        queries = non_empty_lines(queries_sql)
        with open(explain_path, "w", encoding="utf-8") as f:
            for sql in queries:
                if not sql.upper().startswith("EXPLAIN "):
                    sql = "EXPLAIN " + sql
                f.write(sql + "\n")

        subquery_path = work_dir / "grouped_output.sql"
        subquery_result_path = work_dir / "grouped_cards.txt"

        config = {
            "EnableStarCE": 1,
            "UseAssignedAdjustRate": 1,
            "UseSubqueryCard": 0,
            "UseSingleTableCard": 1,
            "RecordingSubquery": 1,
            "SubqueryOutputGroupByMain": 1,
            "RecordingSingleQuery": 0,
            "RefreshStatistics": 0,
            "EnableStarSplit": 0,
            "PredMethod": 1,
            "IsCollectingRelErr": 0,
            "CollectParallel": 8,
            "CompressPrecision": 1.5,
            "SCHEMA_PATH": str(schema_path),
            "SUBQUERY_PATH": str(subquery_path),
            "SUBQUERY_RESULT_PATH": str(subquery_result_path),
            "SINGLE_QUERY_PATH": str(single_query_path),
            "SINGLE_QUERY_RESULT_PATH": str(single_query_result_path),
            "DB_PATH": str(db_path),
            "STATS_PATH": str(stats_path),
            "SQL_PATH": str(explain_path),
            "REAL_CARD_PATH": str(subquery_result_path),
            "REL_ERR_PATH": str(subquery_result_path),
            "ADJUST_RATE": 1,
            "PREDICATE_ADJUST_RATE": 1,
        }

        with open(work_dir / "config.json", "w", encoding="utf-8") as f:
            json.dump(config, f, indent=4)

        print(f"[generate_jobm_grouped_file] Running StarCE with {len(queries)} main queries …")
        t0 = time.time()
        result = subprocess.run(
            [str(starce_binary)],
            cwd=str(work_dir),
            capture_output=True,
            text=True,
            timeout=600,
        )
        elapsed = time.time() - t0
        print(f"[generate_jobm_grouped_file] StarCE finished in {elapsed:.1f}s (rc={result.returncode})")

        if result.returncode != 0:
            stderr_tail = result.stderr[-3000:] if len(result.stderr) > 3000 else result.stderr
            stdout_tail = result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout
            raise RuntimeError(
                f"StarCE exited with code {result.returncode}\n"
                f"STDERR:\n{stderr_tail}\n"
                f"STDOUT:\n{stdout_tail}"
            )

        if not subquery_path.is_file():
            raise RuntimeError(f"StarCE did not produce grouped output at {subquery_path}")

        shutil.copyfile(subquery_path, output_path)
        print(f"[generate_jobm_grouped_file] Grouped file written to {output_path} "
              f"({count_non_empty_lines(output_path)} lines)")
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)

    return output_path


def parse_jobm_grouped_file(grouped_file: Path, main_queries_dir: Path, subquery_output: Path, mapping_output: Path) -> dict:
    grouped_file = require_file(grouped_file)
    main_queries_dir.mkdir(parents=True, exist_ok=True)
    subquery_output.parent.mkdir(parents=True, exist_ok=True)
    mapping_output.parent.mkdir(parents=True, exist_ok=True)

    main_queries = {}
    subqueries = []
    sub_to_main = []
    current_main_id = None

    with open(grouped_file, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.rstrip()
            if not line:
                continue
            if line.startswith("==="):
                parts = line.replace("=", " ").split()
                if len(parts) != 1 or not parts[0].isdigit():
                    raise ValueError(f"Unexpected grouped marker line: {line}")
                current_main_id = int(parts[0])
                continue
            if line.startswith("EXPLAIN "):
                if current_main_id is None:
                    raise ValueError("Encountered EXPLAIN line before grouped id")
                main_sql = line.replace("EXPLAIN ", "", 1).strip()
                if not main_sql.endswith(";"):
                    main_sql += ";"
                main_queries[current_main_id] = main_sql
                with open(main_queries_dir / f"{current_main_id}.sql", "w", encoding="utf-8") as mf:
                    mf.write(main_sql)
                continue
            if line.startswith("SELECT COUNT"):
                sql = line.strip()
                if not sql.endswith(";"):
                    sql += ";"
                subqueries.append(sql)
                sub_to_main.append(current_main_id)

    if not subqueries:
        raise RuntimeError(f"No subqueries parsed from grouped file: {grouped_file}")
    if len(subqueries) != len(sub_to_main):
        raise RuntimeError("JOBM grouped parse produced inconsistent mapping length")
    if any(main_id is None for main_id in sub_to_main):
        raise RuntimeError("Found JOBM subquery without main query mapping")

    with open(subquery_output, "w", encoding="utf-8") as f:
        for sql in subqueries:
            f.write(sql + "\n")

    with open(mapping_output, "wb") as f:
        pickle.dump(sub_to_main, f, pickle.HIGHEST_PROTOCOL)

    return {
        "main_query_count": len(main_queries),
        "grouped_subquery_count": len(subqueries),
        "subquery_output": str(subquery_output),
        "mapping_output": str(mapping_output),
    }


def get_jobm_default_bins() -> dict[str, int]:
    return {
        "title.id": 800,
        "info_type.id": 100,
        "keyword.id": 100,
        "company_name.id": 100,
        "name.id": 100,
        "company_type.id": 100,
        "comp_cast_type.id": 50,
        "kind_type.id": 50,
        "char_name.id": 50,
        "role_type.id": 50,
        "link_type.id": 50,
    }


def train_jobm_model(model_path: Path, data_path: str, model_folder: Path, sample_size: int, seed: int) -> dict:
    np.random.seed(seed)
    n_bins = get_jobm_default_bins()
    schema, table_buckets, ground_truth_factors_no_filter, bins, equivalent_keys = process_imdb_data(
        data_path,
        str(model_folder),
        n_bins,
        "fixed_start_key",
        sample_size=sample_size,
        save_bucket_bins=False,
        seed=seed,
        dataset="jobm",
    )
    bound_ensemble = Bound_ensemble(table_buckets, schema, 1, ground_truth_factors_no_filter)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    with open(model_path, "wb") as f:
        pickle.dump(bound_ensemble, f, pickle.HIGHEST_PROTOCOL)
    return {
        "model_path": str(model_path),
        "bins": bins,
        "equivalent_keys": equivalent_keys,
    }


def sampled_table_name(table_name: str, sampling_type: str, sampling_percentage: float) -> str:
    suffix = f"{sampling_type}{str(sampling_percentage).replace('.', 'd')}"
    return f"{table_name}_{suffix}"


def create_jobm_predicate_indexes(db_conn_kwargs: str, sampling_percentage: float, sampling_type: str) -> None:
    title_table = sampled_table_name("title", sampling_type, sampling_percentage)
    aka_title_table = sampled_table_name("aka_title", sampling_type, sampling_percentage)
    movie_companies_table = sampled_table_name("movie_companies", sampling_type, sampling_percentage)
    movie_info_idx_table = sampled_table_name("movie_info_idx", sampling_type, sampling_percentage)
    movie_info_table = sampled_table_name("movie_info", sampling_type, sampling_percentage)
    movie_keyword_table = sampled_table_name("movie_keyword", sampling_type, sampling_percentage)

    index_sqls = [
        f"CREATE INDEX IF NOT EXISTS idx_{title_table}_prod_year ON {title_table}(production_year);",
        f"CREATE INDEX IF NOT EXISTS idx_{title_table}_kind_id ON {title_table}(kind_id);",
        f"CREATE INDEX IF NOT EXISTS idx_{aka_title_table}_kind_id ON {aka_title_table}(kind_id);",
        f"CREATE INDEX IF NOT EXISTS idx_{movie_companies_table}_ctid ON {movie_companies_table}(company_type_id);",
        f"CREATE INDEX IF NOT EXISTS idx_{movie_info_idx_table}_itid ON {movie_info_idx_table}(info_type_id);",
        f"CREATE INDEX IF NOT EXISTS idx_{movie_info_table}_itid ON {movie_info_table}(info_type_id);",
        f"CREATE INDEX IF NOT EXISTS idx_{movie_keyword_table}_kid ON {movie_keyword_table}(keyword_id);",
    ]

    con = pg.connect(db_conn_kwargs)
    try:
        cursor = con.cursor()
        for sql in index_sqls:
            cursor.execute(sql)
            con.commit()
    finally:
        con.close()


def prepare_jobm_sample_tables(bins, equivalent_keys, db_conn_kwargs: str, sampling_percentage: float, sampling_type: str) -> None:
    create_binned_cols(db_conn_kwargs, bins, equivalent_keys, sampling_percentage, sampling_type)
    create_jobm_predicate_indexes(db_conn_kwargs, sampling_percentage, sampling_type)


def materialize_jobm_samples(query_dir: Path, save_dir: Path, data_path: str, db_conn_kwargs: str, sampling_percentage: float) -> dict:
    require_dir(query_dir)
    schema = gen_jobm_schema(data_path)
    _, equivalent_keys = identify_key_values(schema)
    get_query_binned_cards(str(query_dir), db_conn_kwargs, equivalent_keys, sampling_percentage, str(save_dir))
    materialized_dir = save_dir / f"binned_cards_{sampling_percentage}"
    require_dir(materialized_dir)
    materialized_files = sorted(materialized_dir.glob("*.pkl"))
    if not materialized_files:
        raise RuntimeError(f"No JOBM materialized samples found in {materialized_dir}")
    return {
        "materialized_dir": str(materialized_dir),
        "materialized_file_count": len(materialized_files),
    }


def estimate_jobm_grouped(model_path: Path, grouped_subquery_file: Path, mapping_file: Path, sample_root: Path, sampling_percentage: float, output_path: Path) -> dict:
    model_path = require_file(model_path)
    grouped_subquery_file = require_file(grouped_subquery_file)
    mapping_file = require_file(mapping_file)
    require_dir(sample_root / f"binned_cards_{sampling_percentage}")

    bound_ensemble = load_bound_ensemble(model_path)
    bound_ensemble.SPERCENTAGE = sampling_percentage
    bound_ensemble.query_sample_location = str(sample_root / "binned_cards_{}/")

    with open(mapping_file, "rb") as f:
        mapping = pickle.load(f)
    subqueries = non_empty_lines(grouped_subquery_file)

    if len(subqueries) != len(mapping):
        raise RuntimeError(f"JOBM grouped subquery count mismatch: {len(subqueries)} != {len(mapping)}")

    # Pre-load all materialized sample files (not counted towards estimation time)
    import Sampling.load_sample as _ls
    _sample_cache = {}
    _qdir = bound_ensemble.query_sample_location.format(bound_ensemble.SPERCENTAGE)
    for _idx, _sql in enumerate(subqueries):
        _main_id = mapping[_idx]
        _fpath = os.path.join(_qdir, f"{_main_id}.pkl")
        if _fpath not in _sample_cache:
            with open(_fpath, "rb") as _f:
                _sample_cache[_fpath] = pickle.load(_f)
    # Inject cache: replace disk reads with preloaded data
    _orig_load = _ls.load_sample_imdb_one_query
    def _cached_load(*args, **kwargs):
        return _sample_cache
    _ls.__dict__['load_sample_imdb_one_query'] = lambda tb, ta, qfn, jk, tkg, sp=1.0, qd="": _sample_cache

    predictions = []
    latencies = []
    for idx, sql in enumerate(subqueries):
        main_id = mapping[idx]
        if main_id is None:
            raise RuntimeError(f"JOBM subquery {idx} has no main query mapping")
        t0 = time.time()
        predictions.append(bound_ensemble.get_cardinality_bound_one(sql, query_name=f"{main_id}.pkl"))
        latencies.append(time.time() - t0)

    _ls.load_sample_imdb_one_query = _orig_load

    dump_predictions(predictions, output_path)
    time_file = output_path.parent / f"estimate_time_JOBM_grouped.txt"
    dump_time_file(latencies, time_file)
    return {
        "grouped_query_count": len(subqueries),
        "prediction_path": str(output_path),
        "time_file": str(time_file),
    }


def remap_jobm_results_strict(grouped_sql_path: Path, grouped_pred_path: Path, target_sql_path: Path, output_path: Path) -> dict:
    grouped_sql_path = require_file(grouped_sql_path)
    grouped_pred_path = require_file(grouped_pred_path)
    target_sql_path = require_file(target_sql_path)

    grouped_sqls = non_empty_lines(grouped_sql_path)
    grouped_preds = [float(line) for line in non_empty_lines(grouped_pred_path)]
    target_sqls = non_empty_lines(target_sql_path)

    if len(grouped_sqls) != len(grouped_preds):
        raise RuntimeError(f"JOBM grouped SQL/prediction count mismatch: {len(grouped_sqls)} != {len(grouped_preds)}")

    sql_to_preds = defaultdict(list)
    for sql, pred in zip(grouped_sqls, grouped_preds):
        sql_to_preds[sql].append(pred)

    missing = [sql for sql in target_sqls if sql not in sql_to_preds]
    if missing:
        preview = "\n".join(missing[:3])
        raise RuntimeError(f"JOBM remap missing {len(missing)} target SQLs. Examples:\n{preview}")

    remapped = [float(np.mean(sql_to_preds[sql])) for sql in target_sqls]
    dump_predictions(remapped, output_path)

    return {
        "grouped_sql_count": len(grouped_sqls),
        "unique_grouped_sql_count": len(sql_to_preds),
        "target_sql_count": len(target_sqls),
        "duplicate_grouped_sql_count": sum(1 for preds in sql_to_preds.values() if len(preds) > 1),
        "output_path": str(output_path),
    }


def remap_jobm_times(grouped_sql_path: Path, grouped_time_path: Path, target_sql_path: Path, output_path: Path) -> dict:
    """Map JOBM grouped subquery estimation times back to target subquery.sql order.
    For the same SQL appearing in multiple grouped instances, times are summed."""
    grouped_sql_path = require_file(grouped_sql_path)
    grouped_time_path = require_file(grouped_time_path)
    target_sql_path = require_file(target_sql_path)

    grouped_sqls = non_empty_lines(grouped_sql_path)
    grouped_times = [float(line) for line in non_empty_lines(grouped_time_path)]
    target_sqls = non_empty_lines(target_sql_path)

    if len(grouped_sqls) != len(grouped_times):
        raise RuntimeError(f"JOBM grouped SQL/time count mismatch: {len(grouped_sqls)} != {len(grouped_times)}")

    sql_to_times = defaultdict(list)
    for sql, t in zip(grouped_sqls, grouped_times):
        sql_to_times[sql].append(t)

    missing = [sql for sql in target_sqls if sql not in sql_to_times]
    if missing:
        preview = "\n".join(missing[:3])
        raise RuntimeError(f"JOBM time remap missing {len(missing)} target SQLs. Examples:\n{preview}")

    remapped = [float(np.mean(sql_to_times[sql])) for sql in target_sqls]
    dump_predictions(remapped, output_path)
    return {
        "grouped_sql_count": len(grouped_sqls),
        "unique_sql_count": len(sql_to_times),
        "target_sql_count": len(target_sqls),
        "output_path": str(output_path),
    }


def assert_line_alignment(sql_path: Path, result_path: Path) -> None:
    sql_count = count_non_empty_lines(sql_path)
    result_count = count_non_empty_lines(result_path)
    if sql_count != result_count:
        raise RuntimeError(f"Line count mismatch: {sql_path} has {sql_count}, {result_path} has {result_count}")


def append_summary(records: list[dict], record: dict) -> pd.DataFrame:
    records.append(record)
    df = pd.DataFrame(records)
    df.to_csv(SUMMARY_CSV_PATH, index=False)
    return df

In [5]:
def run_stats_factorjoin() -> dict:
    require_file(STATS_CFG["query_file"])

    _, train_time = measure_call(
        train_one_stats,
        dataset=STATS_CFG["dataset"],
        data_path=STATS_CFG["data_path"],
        model_folder=str(FACTORJOIN_CHECKPOINT_DIR),
        n_dim_dist=STATS_CFG["n_dim_dist"],
        n_bins=STATS_CFG["n_bins"],
        bucket_method=STATS_CFG["bucket_method"],
        save_bucket_bins=False,
        get_bin_means=STATS_CFG["get_bin_means"],
        seed=SEED,
    )

    _, estimate_time = measure_call(
        estimate_bn_queries,
        STATS_CFG["model_path"],
        STATS_CFG["query_file"],
        STATS_CFG["raw_output"],
    )

    copy_result_file(STATS_CFG["raw_output"], STATS_CFG["checkpoint_output"])
    assert_line_alignment(STATS_CFG["query_file"], STATS_CFG["checkpoint_output"])

    return {
        "benchmark": "STATS",
        "train_time_sec": train_time,
        "prepare_sample_time_sec": 0.0,
        "materialize_sample_time_sec": 0.0,
        "estimate_time_sec": estimate_time,
        "total_build_like_time_sec": train_time,
        "total_eval_like_time_sec": estimate_time,
        "query_count": count_non_empty_lines(STATS_CFG["query_file"]),
        "checkpoint_output": str(STATS_CFG["checkpoint_output"]),
        "model_path": str(STATS_CFG["model_path"]),
        "statistics_size_bytes": file_size_bytes(STATS_CFG["model_path"]),
    }


def run_joblight_factorjoin() -> dict:
    require_file(JOBLIGHT_CFG["query_file"])

    _, train_time = measure_call(
        train_one_stats,
        dataset=JOBLIGHT_CFG["dataset"],
        data_path=JOBLIGHT_CFG["data_path"],
        model_folder=str(FACTORJOIN_CHECKPOINT_DIR),
        n_dim_dist=JOBLIGHT_CFG["n_dim_dist"],
        n_bins=JOBLIGHT_CFG["n_bins"],
        bucket_method=JOBLIGHT_CFG["bucket_method"],
        save_bucket_bins=False,
        get_bin_means=JOBLIGHT_CFG["get_bin_means"],
        seed=SEED,
    )

    _, estimate_time = measure_call(
        estimate_bn_queries,
        JOBLIGHT_CFG["model_path"],
        JOBLIGHT_CFG["query_file"],
        JOBLIGHT_CFG["raw_output"],
    )

    copy_result_file(JOBLIGHT_CFG["raw_output"], JOBLIGHT_CFG["checkpoint_output"])
    assert_line_alignment(JOBLIGHT_CFG["query_file"], JOBLIGHT_CFG["checkpoint_output"])

    return {
        "benchmark": "JOBLight",
        "train_time_sec": train_time,
        "prepare_sample_time_sec": 0.0,
        "materialize_sample_time_sec": 0.0,
        "estimate_time_sec": estimate_time,
        "total_build_like_time_sec": train_time,
        "total_eval_like_time_sec": estimate_time,
        "query_count": count_non_empty_lines(JOBLIGHT_CFG["query_file"]),
        "checkpoint_output": str(JOBLIGHT_CFG["checkpoint_output"]),
        "model_path": str(JOBLIGHT_CFG["model_path"]),
        "statistics_size_bytes": file_size_bytes(JOBLIGHT_CFG["model_path"]),
    }


def run_joblightranges_factorjoin() -> dict:
    require_file(JOBLIGHTRANGES_CFG["raw_query_file"])
    require_file(JOBLIGHTRANGES_CFG["true_card_file"])

    JOBLIGHTRANGES_CFG["preprocessed_dir"].mkdir(parents=True, exist_ok=True)
    JOBLIGHTRANGES_CFG["model_dir"].mkdir(parents=True, exist_ok=True)

    _, preprocess_time = measure_call(
        preprocess_imdb_light_data,
        JOBLIGHTRANGES_CFG["raw_data_path"],
        str(JOBLIGHTRANGES_CFG["raw_query_file"]),
        str(JOBLIGHTRANGES_CFG["preprocessed_dir"]),
    )

    require_file(JOBLIGHTRANGES_CFG["preprocessed_query_file"])

    _, train_time = measure_call(
        train_one_stats,
        dataset=JOBLIGHTRANGES_CFG["dataset"],
        data_path=str(JOBLIGHTRANGES_CFG["preprocessed_dir"] / "{}.csv"),
        model_folder=str(JOBLIGHTRANGES_CFG["model_dir"]),
        n_dim_dist=JOBLIGHTRANGES_CFG["n_dim_dist"],
        n_bins=JOBLIGHTRANGES_CFG["n_bins"],
        bucket_method=JOBLIGHTRANGES_CFG["bucket_method"],
        save_bucket_bins=False,
        get_bin_means=JOBLIGHTRANGES_CFG["get_bin_means"],
        seed=SEED,
    )

    _, estimate_time = measure_call(
        estimate_bn_queries,
        JOBLIGHTRANGES_CFG["model_path"],
        JOBLIGHTRANGES_CFG["preprocessed_query_file"],
        JOBLIGHTRANGES_CFG["raw_output"],
    )

    copy_result_file(JOBLIGHTRANGES_CFG["raw_output"], JOBLIGHTRANGES_CFG["checkpoint_output"])
    assert_line_alignment(JOBLIGHTRANGES_CFG["preprocessed_query_file"], JOBLIGHTRANGES_CFG["checkpoint_output"])

    return {
        "benchmark": "JOBLightRanges",
        "preprocess_time_sec": preprocess_time,
        "train_time_sec": train_time,
        "prepare_sample_time_sec": 0.0,
        "materialize_sample_time_sec": 0.0,
        "estimate_time_sec": estimate_time,
        "total_build_like_time_sec": preprocess_time + train_time,
        "total_eval_like_time_sec": estimate_time,
        "query_count": count_non_empty_lines(JOBLIGHTRANGES_CFG["preprocessed_query_file"]),
        "checkpoint_output": str(JOBLIGHTRANGES_CFG["checkpoint_output"]),
        "model_path": str(JOBLIGHTRANGES_CFG["model_path"]),
        "statistics_size_bytes": file_size_bytes(JOBLIGHTRANGES_CFG["model_path"]),
        "preprocessed_query_file": str(JOBLIGHTRANGES_CFG["preprocessed_query_file"]),
    }



def run_statsjoin_factorjoin() -> dict:
    """StatsJoin BN mode evaluation: reuse stats model for BN inference on predicate-free subqueries."""
    require_file(STATSJOIN_CFG["query_file"])

    # No training needed: directly reuse model_stats_greedy_200.pkl
    train_time = 0.0

    _, estimate_time = measure_call(
        estimate_bn_queries,
        STATSJOIN_CFG["model_path"],
        STATSJOIN_CFG["query_file"],
        STATSJOIN_CFG["raw_output"],
    )

    copy_result_file(STATSJOIN_CFG["raw_output"], STATSJOIN_CFG["checkpoint_output"])
    assert_line_alignment(STATSJOIN_CFG["query_file"], STATSJOIN_CFG["checkpoint_output"])

    return {
        "benchmark": "StatsJoin",
        "train_time_sec": train_time,
        "prepare_sample_time_sec": 0.0,
        "materialize_sample_time_sec": 0.0,
        "estimate_time_sec": estimate_time,
        "total_build_like_time_sec": train_time,
        "total_eval_like_time_sec": estimate_time,
        "query_count": count_non_empty_lines(STATSJOIN_CFG["query_file"]),
        "checkpoint_output": str(STATSJOIN_CFG["checkpoint_output"]),
        "model_path": str(STATSJOIN_CFG["model_path"]),
        "statistics_size_bytes": file_size_bytes(STATSJOIN_CFG["model_path"]),
    }

def run_jobm_factorjoin(force_reparse: bool = True) -> dict:
    if not JOBM_CFG["grouped_file"].exists():
        print(f"\n[JOBM] grouped_file not found, generating with StarCE …")
        generate_jobm_grouped_file(
            queries_sql=JOBM_CFG["queries_sql"],
            output_path=JOBM_CFG["grouped_file"],
            starce_binary=JOBM_CFG["starce_binary"],
            schema_path=JOBM_CFG["schema_path"],
            db_path=JOBM_CFG["db_path"],
            stats_path=JOBM_CFG["stats_path"],
            single_query_path=JOBM_CFG["single_query_path"],
            single_query_result_path=JOBM_CFG["single_query_result_path"],
        )

    require_file(JOBM_CFG["grouped_file"])
    require_file(JOBM_CFG["target_subquery_file"])

    parse_time = 0.0
    if force_reparse or not JOBM_CFG["grouped_subquery_file"].exists() or not JOBM_CFG["mapping_file"].exists():
        _, parse_time = measure_call(
            parse_jobm_grouped_file,
            JOBM_CFG["grouped_file"],
            JOBM_CFG["main_queries_dir"],
            JOBM_CFG["grouped_subquery_file"],
            JOBM_CFG["mapping_file"],
        )

    jobm_train_meta, train_time = measure_call(
        train_jobm_model,
        JOBM_CFG["model_path"],
        JOBM_CFG["data_path"],
        FACTORJOIN_CHECKPOINT_DIR,
        JOBM_CFG["sample_size"],
        JOBM_CFG["seed"],
    )

    _, prepare_sample_time = measure_call(
        prepare_jobm_sample_tables,
        jobm_train_meta["bins"],
        jobm_train_meta["equivalent_keys"],
        JOBM_CFG["db_conn_kwargs"],
        JOBM_CFG["sampling_percentage"],
        JOBM_CFG["sampling_type"],
    )

    _, materialize_sample_time = measure_call(
        materialize_jobm_samples,
        JOBM_CFG["main_queries_dir"],
        FACTORJOIN_CHECKPOINT_DIR,
        JOBM_CFG["data_path"],
        JOBM_CFG["db_conn_kwargs"],
        JOBM_CFG["sampling_percentage"],
    )

    _, estimate_time = measure_call(
        estimate_jobm_grouped,
        JOBM_CFG["model_path"],
        JOBM_CFG["grouped_subquery_file"],
        JOBM_CFG["mapping_file"],
        FACTORJOIN_CHECKPOINT_DIR,
        JOBM_CFG["sampling_percentage"],
        JOBM_CFG["grouped_output"],
    )

    remap_info = remap_jobm_results_strict(
        JOBM_CFG["grouped_subquery_file"],
        JOBM_CFG["grouped_output"],
        JOBM_CFG["target_subquery_file"],
        JOBM_CFG["checkpoint_output"],
    )
    assert_line_alignment(JOBM_CFG["target_subquery_file"], JOBM_CFG["checkpoint_output"])

    # Remap estimate time from grouped to target subquery order
    grouped_time_path = CHECKPOINT_DIR / "estimate_time_JOBM_grouped.txt"
    if grouped_time_path.exists():
        time_info = remap_jobm_times(
            JOBM_CFG["grouped_subquery_file"],
            grouped_time_path,
            JOBM_CFG["target_subquery_file"],
            JOBM_CFG["checkpoint_output"].parent / "estimate_time_JOBM.txt",
        )

    return {
        "benchmark": "JOBM",
        "parse_grouped_time_sec": parse_time,
        "train_time_sec": train_time,
        "prepare_sample_time_sec": prepare_sample_time,
        "materialize_sample_time_sec": materialize_sample_time,
        "estimate_time_sec": estimate_time,
        "total_build_like_time_sec": train_time + prepare_sample_time,
        "total_eval_like_time_sec": materialize_sample_time + estimate_time,
        "grouped_query_count": count_non_empty_lines(JOBM_CFG["grouped_subquery_file"]),
        "query_count": count_non_empty_lines(JOBM_CFG["target_subquery_file"]),
        "checkpoint_output": str(JOBM_CFG["checkpoint_output"]),
        "grouped_output": str(JOBM_CFG["grouped_output"]),
        "model_path": str(JOBM_CFG["model_path"]),
        "statistics_size_bytes": file_size_bytes(JOBM_CFG["model_path"]) + dir_size_bytes(FACTORJOIN_CHECKPOINT_DIR / f"binned_cards_{JOBM_SAMPLING_PERCENTAGE}"),
        **remap_info,
    }


In [6]:
from Evaluation.training import train_one_imdb
from Evaluation.testing import test_on_jobjoin


def prepare_jobjoin_inputs(queries_sql: Path, main_queries_dir: Path, clean_file: Path, mapping_file: Path) -> dict:
    """Split JobJoin queries.sql (31 pure joins) into per-query .sql + clean query file + identity mapping."""
    queries_sql = require_file(queries_sql)
    main_queries_dir = Path(main_queries_dir)
    main_queries_dir.mkdir(parents=True, exist_ok=True)
    clean_file = Path(clean_file)
    clean_file.parent.mkdir(parents=True, exist_ok=True)

    queries = non_empty_lines(queries_sql)
    mapping = []
    with open(clean_file, "w", encoding="utf-8") as cf:
        for i, sql in enumerate(queries, start=1):
            main_id = str(i)
            with open(main_queries_dir / f"{main_id}.sql", "w", encoding="utf-8") as mf:
                mf.write(sql)
            cf.write(sql + "\n")
            mapping.append(main_id)
    with open(mapping_file, "wb") as f:
        pickle.dump(mapping, f, pickle.HIGHEST_PROTOCOL)
    return {"main_query_count": len(queries)}


def run_jobjoin_factorjoin() -> dict:
    cfg = JOBJOIN_CFG
    require_file(cfg["queries_sql"])

    # 1. Prepare inputs (split main queries + identity mapping)
    _, prepare_input_time = measure_call(
        prepare_jobjoin_inputs,
        cfg["queries_sql"],
        cfg["main_queries_dir"],
        cfg["clean_query_file"],
        cfg["mapping_file"],
    )

    # 2. Delete gt_no_filter.pkl cache potentially contaminated by JOBM (no dataset name, shared by jobm 17 tables / imdb 21 tables).
    #    Not deleting would cause imdb model to miss tables like name/person_info, evaluation raises KeyError: 'name'.
    if cfg["gt_cache"].exists():
        cfg["gt_cache"].unlink()

    # 3. Train full imdb model + materialize samples (no predicates -> prepare_sample=False does not create sample tables; materialize all None, no PG connection)
    _, train_time = measure_call(
        train_one_imdb,
        cfg["data_path"],
        str(FACTORJOIN_CHECKPOINT_DIR),
        cfg["n_dim_dist"],
        cfg["n_bins"],
        cfg["bucket_method"],
        cfg["sample_size"],
        None,    # external_workload_file
        False,   # save_bucket_bins
        cfg["seed"],
        False,   # prepare_sample
        cfg["db_conn_kwargs"],
        cfg["sampling_percentage"],
        cfg["sampling_type"],
        str(cfg["main_queries_dir"]),  # test_query_file（trigger materialization，traverse by directory）
        True,    # materialize_sample
        dataset="imdb",
    )

    # 4. Evaluate (reuse fixed get_cardinality_bound_one; cyclic self-join Q31 -> MISSING)
    sample_loc = str(FACTORJOIN_CHECKPOINT_DIR / "binned_cards_{}/")
    _, estimate_time = measure_call(
        test_on_jobjoin,
        str(cfg["model_path"]),
        str(cfg["clean_query_file"]),
        str(cfg["mapping_file"]),
        cfg["sampling_percentage"],
        sample_loc,
        str(cfg["result_output"]),
    )

    # 5. Copy to checkpoint directory, consistent with other benchmarks; alignment check (with MISSING row still 31 lines)
    copy_result_file(cfg["result_output"], cfg["checkpoint_output"])
    assert_line_alignment(cfg["queries_sql"], cfg["result_output"])

    return {
        "benchmark": "JobJoin",
        "prepare_input_time_sec": prepare_input_time,
        "train_time_sec": train_time,  # includes materialization（train_one_imdb internal materialize_sample）
        "prepare_sample_time_sec": 0.0,
        "materialize_sample_time_sec": 0.0,
        "estimate_time_sec": estimate_time,
        "total_build_like_time_sec": train_time,
        "total_eval_like_time_sec": estimate_time,
        "query_count": count_non_empty_lines(cfg["queries_sql"]),
        "checkpoint_output": str(cfg["checkpoint_output"]),
        "result_output": str(cfg["result_output"]),
        "model_path": str(cfg["model_path"]),
        "statistics_size_bytes": file_size_bytes(cfg["model_path"]),
    }


## Execute

Running the next cell will:
- train and evaluate `STATS`
- train and evaluate `JOBLight`
- preprocess, train, and evaluate `JOBLightRanges`
- parse grouped `JOBM`, train it, prepare sample tables, materialize query samples, evaluate grouped subqueries, and remap them back to `subquery.sql` order

The summary table is saved to `experiment/checkpoint/FactorJoin/benchmark_times.csv` after each benchmark finishes.


In [7]:
records = []
summary_df = pd.DataFrame()

for benchmark in BENCHMARKS_TO_RUN:
    print(f"\n{'=' * 80}")
    print(f"Running {benchmark}")
    print(f"{'=' * 80}")

    if benchmark == "STATS":
        record = run_stats_factorjoin()
    elif benchmark == "JOBLight":
        record = run_joblight_factorjoin()
    elif benchmark == "JOBLightRanges":
        record = run_joblightranges_factorjoin()
    elif benchmark == "JOBM":
        record = run_jobm_factorjoin(force_reparse=FORCE_REPARSE_JOBM_GROUPED)
    elif benchmark == "JobJoin":
        record = run_jobjoin_factorjoin()
    elif benchmark == "StatsJoin":
        record = run_statsjoin_factorjoin()
    else:
        raise ValueError(f"Unsupported benchmark: {benchmark}")

    summary_df = append_summary(records, record)
    display(summary_df.tail(1))

summary_df



Running STATS
bucketizing equivalent key group: {'posts.Id', 'postLinks.PostId', 'tags.ExcerptPostId', 'votes.PostId', 'comments.PostId', 'postLinks.RelatedPostId', 'postHistory.PostId'}
bucketizing equivalent key group: {'posts.OwnerUserId', 'postHistory.UserId', 'users.Id', 'comments.UserId', 'badges.UserId', 'votes.UserId'}


INFO:BayesCard.Models.BN_single_model:Discretizing table takes 0.016399621963500977 secs
INFO:BayesCard.Models.BN_single_model:Learning BN optimal structure from data with 79851 rows and 2 cols


badges
{'badges.UserId': 'categorical', 'badges.Date': 'continuous'}
Discretizing table takes 0.017919063568115234 secs


INFO:BayesCard.Models.BN_single_model:Structure learning took 0.5884156227111816 secs.
INFO:BayesCard.Models.Bayescard_BN:Model spec[('badges.UserId', 'badges.Date')]
INFO:BayesCard.Models.Bayescard_BN:calling pgm.BayesianModel.fit...
INFO:BayesCard.Models.Bayescard_BN:done, took 0.02971792221069336 secs.


Structure learning took 0.5896022319793701 secs.
done, parameter learning took 0.030783414840698242 secs.
votes
{'votes.PostId': 'categorical', 'votes.VoteTypeId': 'categorical', 'votes.CreationDate': 'continuous', 'votes.UserId': 'categorical', 'votes.BountyAmount': 'categorical'}


...


Running JobJoin


/home/liwei/starce-final/StarCE/methods/FactorJoin/Join_scheme/data_prepare.py:673: DtypeWarning: Columns (5,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rows = pd.read_csv(table_obj.csv_file_location, header=None, escapechar='\\', encoding='utf-8',
/home/liwei/starce-final/StarCE/methods/FactorJoin/Join_scheme/data_prepare.py:673: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rows = pd.read_csv(table_obj.csv_file_location, header=None, escapechar='\\', encoding='utf-8',
/home/liwei/starce-final/StarCE/methods/FactorJoin/Join_scheme/data_prepare.py:673: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rows = pd.read_csv(table_obj.csv_file_location, header=None, escapechar='\\', encoding='utf-8',
/home/liwei/starce-final/StarCE/methods/FactorJoin/Join_scheme/data_prepare.py:673: DtypeWarning: Columns (4) have mixed types. Specify dtype 

title.kind_id
movie_info.info_type_id
movie_keyword.movie_id
cast_info.person_id
cast_info.person_role_id
cast_info.role_id
complete_cast.status_id
movie_link.link_type_id
movie_keyword.keyword_id
movie_companies.company_id
movie_companies.company_type_id
kind_type.id 8
info_type.id 72
title.id 74
name.id 27
char_name.id 28
role_type.id 12
comp_cast_type.id 2
link_type.id 17
keyword.id 86
company_name.id 75
company_type.id 3
models save at /home/liwei/starce-final/StarCE/methods/FactorJoin/checkpoints/model_imdb_default.pkl
1
5.0067901611328125e-06 2.384185791015625e-07
10
2.6226043701171875e-06 2.384185791015625e-07
11
5.245208740234375e-06 2.384185791015625e-07
12
5.245208740234375e-06 2.384185791015625e-07
13
4.76837158203125e-06 2.384185791015625e-07
14
5.4836273193359375e-06 2.384185791015625e-07
15
5.4836273193359375e-06 2.384185791015625e-07
16
5.245208740234375e-06 2.384185791015625e-07
17
5.4836273193359375e-06 2.384185791015625e-07
18
9.059906005859375e-06 2.384185791015625e-

,benchmark,train_time_sec,prepare_sample_time_sec,materialize_sample_time_sec,estimate_time_sec,total_build_like_time_sec,total_eval_like_time_sec,query_count,checkpoint_output,model_path,...,parse_grouped_time_sec,grouped_query_count,grouped_output,grouped_sql_count,unique_grouped_sql_count,target_sql_count,duplicate_grouped_sql_count,output_path,prepare_input_time_sec,result_output
5,JobJoin,1534.378835,0.0,0.0,0.364549,1534.378835,0.364549,31,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002127,/home/liwei/starce-final/StarCE/Benchmark/work...


,benchmark,train_time_sec,prepare_sample_time_sec,materialize_sample_time_sec,estimate_time_sec,total_build_like_time_sec,total_eval_like_time_sec,query_count,checkpoint_output,model_path,...,parse_grouped_time_sec,grouped_query_count,grouped_output,grouped_sql_count,unique_grouped_sql_count,target_sql_count,duplicate_grouped_sql_count,output_path,prepare_input_time_sec,result_output
0,STATS,87.502290,0.000000,0.000000,30.742124,87.502290,30.742124,2471,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JOBLight,5492.079054,0.000000,0.000000,2.074946,5492.079054,2.074946,451,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JOBLightRanges,2608.655281,0.000000,0.000000,37.392125,2757.440866,37.392125,8292,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JOBM,233.978487,124.470882,140.787651,43.160965,358.449368,183.948616,6424,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,0.348885,9472.0,/home/liwei/starce-final/StarCE/experiment/che...,9472.0,6424.0,6424.0,1639.0,/home/liwei/starce-final/StarCE/experiment/che...,NaN,NaN
4,StatsJoin,0.000000,0.000000,0.000000,3.074106,0.000000,3.074106,226,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,JobJoin,1534.378835,0.000000,0.000000,0.364549,1534.378835,0.364549,31,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002127,/home/liwei/starce-final/StarCE/Benchmark/work...


In [8]:
summary_df = pd.read_csv(SUMMARY_CSV_PATH)
summary_df


,benchmark,train_time_sec,prepare_sample_time_sec,materialize_sample_time_sec,estimate_time_sec,total_build_like_time_sec,total_eval_like_time_sec,query_count,checkpoint_output,model_path,...,parse_grouped_time_sec,grouped_query_count,grouped_output,grouped_sql_count,unique_grouped_sql_count,target_sql_count,duplicate_grouped_sql_count,output_path,prepare_input_time_sec,result_output
0,STATS,87.502290,0.000000,0.000000,30.742124,87.502290,30.742124,2471,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JOBLight,5492.079054,0.000000,0.000000,2.074946,5492.079054,2.074946,451,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JOBLightRanges,2608.655281,0.000000,0.000000,37.392125,2757.440866,37.392125,8292,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JOBM,233.978487,124.470882,140.787651,43.160965,358.449368,183.948616,6424,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,0.348885,9472.0,/home/liwei/starce-final/StarCE/experiment/che...,9472.0,6424.0,6424.0,1639.0,/home/liwei/starce-final/StarCE/experiment/che...,NaN,NaN
4,StatsJoin,0.000000,0.000000,0.000000,3.074106,0.000000,3.074106,226,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,JobJoin,1534.378835,0.000000,0.000000,0.364549,1534.378835,0.364549,31,/home/liwei/starce-final/StarCE/experiment/che...,/home/liwei/starce-final/StarCE/methods/Factor...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002127,/home/liwei/starce-final/StarCE/Benchmark/work...
